# Table 2 Reproduction Audit

This notebook reruns the repository training entry points used for the
Inventory and Cliff-Walking experiments, then compares the local results
with the numerical targets available from the paper.

The notebook does not modify repository source files. The helper
scripts under `notebooks/reproduction` only assemble commands, clear the
selected output directory when requested, call the original repo modules,
and summarize the generated `results.json` files.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path.cwd()
if REPO.name != "State_Aggregation_Project":
    REPO = Path(r"D:\python_project\git\State_Aggregation_Project")

CONDA_ENV_PYTHON = Path(r"C:\ProgramData\anaconda3\envs\dfl_sa_cuda\python.exe")
PYTHON = str(CONDA_ENV_PYTHON if CONDA_ENV_PYTHON.exists() else Path(sys.executable))
SEEDS = [1,2,3,4,5,6,7,8,9,10]

INVENTORY_ROOT = REPO / "notebooks" / "reproduction" / "inventory_table2_10seed_runs"
CLIFF_ROOT = REPO / "notebooks" / "reproduction" / "cliff_dist_shift_table2_10seed"

# Keep this True for a clean reproduction run. Set False only while
# editing text or plots after the full run has already completed.
RUN_FULL_TRAINING = True


## 1. Paper Parameters Used Here

The table below is the part of Table 2 that is directly relevant to the
two reproduced experiments.


In [ ]:
table2 = pd.DataFrame([
    {
        "task": "Inventory",
        "state/action": "|S|=36, |A|=31",
        "discount": 0.95,
        "architecture": "MLP (32, 32), ReLU",
        "optimizer": "Adam, lr=1e-2, full batch",
        "pretrain/train": "30 / 50 epochs",
        "seeds": 10,
        "surrogate": "LP rho=10, Sketch rho=0.1, QP rho=1e4, eta=1e-3",
        "sketch": "K=8, alpha=0.1",
        "aggregation": "M in {10,20,30}, tau=0.5",
        "data": "train=10, test=50 in paper",
    },
    {
        "task": "Cliff Walking",
        "state/action": "|S|=48, |A|=4",
        "discount": 0.95,
        "architecture": "MLP (64, 64), ReLU + Tanh",
        "optimizer": "Adam, lr=3e-3, batch=32",
        "pretrain/train": "60 / 50 epochs",
        "seeds": 10,
        "surrogate": "LP rho=0.1, Sketch rho=1.0, QP rho=0.1, eta=0.1",
        "sketch": "K=8, alpha=0.2",
        "aggregation": "M in {10,20,30}, tau=0.5",
        "data": "train=100, test=100",
    },
])
display(table2)


## 2. Inventory Reproduction

The command below calls `mdp_lp.run_elasticity_experiments` through the
notebook-local runner. It uses the Inventory architecture and optimization
settings from Table 2 where the current CLI exposes them.

One important repository detail is checked after the run: the current
`mdp_lp` data code multiplies `--n_samples` by four and then uses a
1/4 vs 3/4 train/test split. Therefore `--n_samples 10` gives 10 train
and 30 test scenarios in the current repo, while Table 2 reports 10 train
and 50 test scenarios.


In [ ]:
inv_cmd = [
    PYTHON,
    str(REPO / "notebooks" / "reproduction" / "run_inventory_figure_recipe.py"),
    "--root", str(INVENTORY_ROOT),
    "--methods", "twostage", "dfl", "qpth", "sketch", "sa10", "sa20", "sa30",
    "--seeds", *map(str, SEEDS),
    "--epochs", "50",
    "--pretrain_epochs", "30",
    "--concurrency", "2",
    "--no_skip_existing",
]

if RUN_FULL_TRAINING:
    if INVENTORY_ROOT.exists():
        shutil.rmtree(INVENTORY_ROOT)
    subprocess.run(inv_cmd, cwd=REPO, check=True)
else:
    print("Skipped full Inventory training. Command that should be run:")
    print(" ".join(map(str, inv_cmd)))


In [ ]:
inv_summary = (INVENTORY_ROOT / "summary.md").read_text(encoding="utf-8")
display(Markdown(inv_summary))

for name in [
    "inventory_training_curves.png",
    "inventory_decision_quality.png",
    "inventory_compute_cost.png",
]:
    path = INVENTORY_ROOT / "paper_style_figures" / name
    display(Markdown(f"**{name}**"))
    display(Image(filename=str(path)))


In [ ]:
sample_cfg = json.loads(
    (INVENTORY_ROOT / "exp_b" / "normal_margin" / "twostage" /
     "tag_seed_1_s30q30" / "epochs_50" / "results.json").read_text(encoding="utf-8")
)
inventory_config_check = {
    "requested_n_samples": sample_cfg.get("requested_n_samples"),
    "generated_n_samples": sample_cfg.get("generated_n_samples"),
    "n_epochs": sample_cfg.get("n_epochs"),
    "pretrain_epochs": sample_cfg.get("pretrain_epochs"),
    "s_max": sample_cfg.get("s_max"),
    "q_max": sample_cfg.get("q_max"),
    "b_max": sample_cfg.get("b_max"),
}
inventory_config_check


## 3. Cliff-Walking Reproduction

Table 2 specifies the optimizer, architecture, seed count, and sample
split, but not the full weather-regime distribution. The repository
contains several Cliff routes. The route that reproduces the paper
numbers for Two-stage and DFL-LP is the dist-shift recipe used below:
`regime_set=v4`, `center_scale=0.88`, `ambiguous_pairs=0:2`,
train regime weights `15,1,1,1`, test regime weights `0,0,1,0`,
`cliff_penalty=-50`, and `noise_std=0.6`.

The same runner also attempts `dfl_sketch`. In the current repo state,
this path fails before completing training because `train_cliff.py`
passes `qp_solver` and `qp_reg` into `mdp_lp.solvers.sketched_lp`, while
that function does not expose those keyword arguments.


In [ ]:
cliff_cmd = [
    PYTHON,
    str(REPO / "notebooks" / "reproduction" / "run_cliff_table2_reproduction.py"),
    "--root", str(CLIFF_ROOT),
    "--routes", "dist_shift_recipe",
    "--methods", "twostage", "dfl", "dfl_qpth", "dfl_sketch", "ssa10", "ssa20", "ssa30",
    "--seeds", *map(str, SEEDS),
    "--concurrency", "2",
    "--fresh",
    "--no_skip_existing",
]

if RUN_FULL_TRAINING:
    subprocess.run(cliff_cmd, cwd=REPO, check=True)
else:
    print("Skipped full Cliff training. Command that should be run:")
    print(" ".join(map(str, cliff_cmd)))


In [ ]:
cliff_summary = (CLIFF_ROOT / "summary.md").read_text(encoding="utf-8")
display(Markdown(cliff_summary))


In [ ]:
failed_log = CLIFF_ROOT / "dist_shift_recipe" / "dfl_sketch_s1" / "run.log"
text = failed_log.read_text(encoding="utf-8", errors="replace")
print(text[-1600:])


## 4. Reproduction Status

**1. Inventory parameter mismatch.** The current checked-out repo has a
tracked `run_paper_repro.py` script, but its Inventory hyperparameters
are not identical to Table 2. The largest confirmed differences are in
the augmented-Lagrangian penalty: Table 2 lists Inventory DFL-LP
`rho=10`, while `run_paper_repro.py` uses `rho=10000`; Table 2 lists
canonical DFL-Sketch `rho=0.1`, while `run_paper_repro.py` uses
`rho=10.0`. These are order-of-magnitude differences, not harmless
formatting differences. Under the Table-2-style route used here, all
Inventory methods run for 10 seeds, but DFL-LP and DFL-QP do not match
the paper-style pattern. The SA trend is still visible: regret improves
as M increases from 10 to 30, and SA is much cheaper than QP.

**2. Inventory data split mismatch.** Table 2 reports 10 train samples
and 50 test samples. In the current `mdp_lp` code, `--n_samples 10`
becomes `generated_n_samples=40`, with a 1/4 train split, so the actual
split is 10 train and 30 test. This is another obstacle to strict
numerical reproduction.

**3. Cliff hyperparameter route is incomplete in the public script.**
The tracked `run_paper_repro.py` does not encode the Table 2 Cliff
configuration completely. Several key values are omitted or differ from
Table 2, including learning rate, QP regularization, Sketch rho, Sketch
ensemble size, Sketch row ratio, and aggregation sizes. Some values
happen to match current CLI defaults, but they are not explicit in that
script.

**4. Cliff-QP is runnable but not numerically matched.** Using the
recovered dist-shift route, Two-stage and DFL-LP reproduce the paper
table closely. DFL-QP runs successfully with Table-2-style `rho=0.1`
and `eta=0.1`, but the local result is `11.0025` versus the paper value
`9.53`. This suggests that additional QP/training details are missing
or have drifted, such as qpth iteration count, tolerance, gradient
handling, loss weighting, or other solver settings.

**5. Cliff-Sketch is blocked by interface drift.** The current
`dfl_sketch` path in `mdp_cw.train_cliff` calls
`sketched_lp(..., qp_solver=..., qp_reg=...)`, but
`mdp_lp.solvers.sketched_lp.sketched_lp` does not accept those keyword
arguments. Therefore Cliff DFL-Sketch fails through the repo CLI before
completing training.
